In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [ ]:
# load dataset

df = pd.read_csv("/content/arxiv_abstracts_and_titles.csv")
print(df.shape)
df.head()

(79141, 7)


,category,field,discipline,title,abstract,abstract_length,title_length
0,cs.LG,Machine Learning,Artificial Intelligence,how do lexical semantics affect translation an...,neural machine translation (nmt) systems aim t...,177,9
1,cs.LG,Machine Learning,Artificial Intelligence,barack partially supervised group robustness w...,while neural networks have shown remarkable su...,203,7
2,cs.LG,Machine Learning,Artificial Intelligence,a deep learning approach to integrate humanlev...,"in recent times, a large number of people have...",187,11
3,cs.LG,Machine Learning,Artificial Intelligence,croesus multistage processing and transactions...,emerging edge applications require both a fast...,194,10
4,cs.LG,Machine Learning,Artificial Intelligence,representation topology divergence a method fo...,comparison of data representations is a comple...,133,10


In [ ]:
# clean data
df = df.copy()

for col in ["title", "abstract", "discipline", "field", "category"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

df = df[(df["title"] != "") | (df["abstract"] != "")]
df = df.reset_index(drop=True)

print("Cleaned shape:",df.shape)

Cleaned shape: (79141, 7)


In [ ]:
# build text column
df["text"] = (
    "title: " + df["title"] +
    " [SEP] abstract: " + df["abstract"]
)

display(df[["title", "abstract", "text"]].head())

,title,abstract,text
0,how do lexical semantics affect translation an...,neural machine translation (nmt) systems aim t...,title: how do lexical semantics affect transla...
1,barack partially supervised group robustness w...,while neural networks have shown remarkable su...,title: barack partially supervised group robus...
2,a deep learning approach to integrate humanlev...,"in recent times, a large number of people have...",title: a deep learning approach to integrate h...
3,croesus multistage processing and transactions...,emerging edge applications require both a fast...,title: croesus multistage processing and trans...
4,representation topology divergence a method fo...,comparison of data representations is a comple...,title: representation topology divergence a me...


In [ ]:
# load tokenizer

model_name ="allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# metrics

def compute_metrics(eval_pred):
    logits,labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1
    }


In [ ]:
# tokenizer

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# training fucntion

def train_scibert_for_target(
    df,
    target_col,
    output_dir,
    cap_per_class=None,
    num_train_epochs=3,
    learning_rate=1e-5,
    train_batch_size=16,
    eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=500,
    random_state=42
):
    work_df = df.copy()

    # remove the empty labels
    work_df = work_df[work_df[target_col] != ""].reset_index(drop=True)

    # cap for class balance
    if cap_per_class is not None:
        work_df = (
            work_df.groupby(target_col, group_keys=False)
            .apply(lambda x: x.sample(n=min(len(x), cap_per_class), random_state=random_state))
            .sample(frac=1, random_state=random_state)
            .reset_index(drop=True)
        )

    print(f"\nTarget: {target_col}")
    print("Shape after filtering & capping:", work_df.shape)
    print(work_df[target_col].value_counts())

    # labels
    label_encoder = LabelEncoder()
    work_df["label"] = label_encoder.fit_transform(work_df[target_col])

    id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
    label2id = {label: i for i, label in id2label.items()}
    num_labels = len(label_encoder.classes_)

    print("\nNumber of labels:", num_labels)
    print(id2label)

    # split
    train_df, val_df = train_test_split(
        work_df[["text", "label", target_col]],
        test_size=0.2,
        random_state= random_state,
        stratify=work_df["label"]
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print("\nTrain:", train_df.shape)
    print("Validation:", val_df.shape)

    # datasets
    train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
    val_ds = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)

    train_ds = train_ds.map(tokenize_function, batched=True)
    val_ds = val_ds.map(tokenize_function, batched=True)

    # model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

    # args
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        fp16=torch.cuda.is_available(),
        save_total_limit=2,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    eval_results = trainer.evaluate()

    print("Evaluation: ")
    for k, v in eval_results.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

    pred_output = trainer.predict(val_ds)
    pred_ids = np.argmax(pred_output.predictions, axis=1)

    val_results = val_df.copy()
    val_results["pred_label"] = pred_ids
    val_results[f"pred_{target_col}"] = label_encoder.inverse_transform(pred_ids)

    print("Report:")
    print(classification_report(
        val_results[target_col],
        val_results[f"pred_{target_col}"],
        zero_division=0
    ))

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    label_map_df = pd.DataFrame({
        "label_id": list(id2label.keys()),
        target_col: list(id2label.values())
    })
    label_map_df.to_csv(f"{output_dir}/label_mapping.csv", index=False)

    print(f"\nModel saved to: {output_dir}")

    return {
        "trainer": trainer,
        "eval_results": eval_results,
        "val_results": val_results,
        "label_encoder": label_encoder
    }

In [ ]:
# discipline
discipline_run = train_scibert_for_target(
    df=df,
    target_col="discipline",
    output_dir="/content/scibert_discipline",
    cap_per_class=2500,
    num_train_epochs=3,
    learning_rate=1e-5,
    train_batch_size=16,
    eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=500
)

/tmp/ipykernel_16909/3829469231.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), cap_per_class), random_state=random_state))



Target: discipline
Shape after filtering & capping: (54426, 8)
discipline
Information Retrieval           2500
Multimedia                      2500
Computer Graphics               2500
Computer Imaging and Vision     2500
Cybersecurity                   2500
Theory of Computation           2500
Communication                   2500
Database Systems                2500
Artificial Intelligence         2500
Computer Hardware               2500
World Wide Web                  2500
Robotics                        2500
Computer Networks               2500
Human-Computer Interaction      2499
Computational Science           2499
Social Computing                2499
Distributed Systems             2498
Software Engineering            2494
Computer Programming            2486
Emerging Technologies           2474
Theoretical Computer Science    2301
Computer Systems                2176
Name: count, dtype: int64

Number of labels: 22
{0: 'Artificial Intelligence', 1: 'Communication', 2: 'Computat

Map:   0%|          | 0/43540 [00:00<?, ? examples/s]

Map:   0%|          | 0/10886 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Evaluation: 
eval_loss: 1.0854
eval_accuracy: 0.6595
eval_macro_precision: 0.6571
eval_macro_recall: 0.6586
eval_macro_f1: 0.6558
eval_weighted_precision: 0.6574
eval_weighted_recall: 0.6595
eval_weighted_f1: 0.6564
eval_runtime: 18.2144
eval_samples_per_second: 597.6600
eval_steps_per_second: 37.3880
epoch: 3.0000
Report:
                              precision    recall  f1-score   support

     Artificial Intelligence       0.50      0.34      0.40       500
               Communication       0.73      0.76      0.75       500
       Computational Science       0.79      0.73      0.76       500
           Computer Graphics       0.83      0.84      0.83       500
           Computer Hardware       0.63      0.69      0.66       500
 Computer Imaging and Vision       0.66      0.67      0.66       500
           Computer Networks       0.61      0.61      0.61       500
        Computer Programming       0.70      0.73      0.71       497
            Computer Systems       0.54     

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: /content/scibert_discipline


In [ ]:
display(discipline_run["val_results"].head())

,text,label,discipline,pred_label,pred_discipline
0,title: a highlevel synthesis toolchain for the...,7,Computer Programming,7,Computer Programming
1,title: continuous offline handwriting recognit...,5,Computer Imaging and Vision,14,Information Retrieval
2,title: agentic multipersona framework for evid...,14,Information Retrieval,17,Social Computing
3,title: lowpower hardwarebased deeplearning dia...,4,Computer Hardware,4,Computer Hardware
4,title: intelligence reflecting surfaceaided in...,6,Computer Networks,1,Communication


In [ ]:
# field

field_run = train_scibert_for_target(
    df=df,
    target_col="field",
    output_dir="/content/scibert_field",
    cap_per_class=None,
    num_train_epochs=3,
    learning_rate=1e-5,
    train_batch_size=16,
    eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=500
)


Target: field
Shape after filtering & capping: (79141, 8)
field
Machine Learning                        2500
Computer Vision                         2500
General Artificial Intelligence         2500
Natural Language Processing             2500
Information Theory                      2500
General Robotics                        2500
Cryptography                            2500
Data Structures and Algorithms          2500
Sound and Signal Processing             2500
General Computer Networks               2500
Social Networks                         2500
General Information Retrieval           2500
General Multimedia                      2500
Database Systems                        2500
Discrete Mathematics                    2500
Neural Networks                         2500
Computational Geometry                  2500
Hardware Architecture                   2500
Computers and Society                   2499
General Human-Computer Interaction      2499
Computational Engineering          

Map:   0%|          | 0/63312 [00:00<?, ? examples/s]

Map:   0%|          | 0/15829 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Evaluation: 
eval_loss: 1.3503
eval_accuracy: 0.5819
eval_macro_precision: 0.5696
eval_macro_recall: 0.5823
eval_macro_f1: 0.5722
eval_weighted_precision: 0.5691
eval_weighted_recall: 0.5819
eval_weighted_f1: 0.5717
eval_runtime: 26.9029
eval_samples_per_second: 588.3760
eval_steps_per_second: 36.7990
epoch: 3.0000
Report:
                                      precision    recall  f1-score   support

            Computational Complexity       0.58      0.56      0.57       499
           Computational Engineering       0.66      0.62      0.64       500
              Computational Geometry       0.71      0.78      0.74       500
                     Computer Vision       0.46      0.58      0.52       500
               Computers and Society       0.43      0.41      0.42       500
                        Cryptography       0.47      0.57      0.52       500
      Data Structures and Algorithms       0.52      0.50      0.51       500
                    Database Systems       0.64   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: /content/scibert_field


In [ ]:
display(field_run["val_results"].head())

,text,label,field,pred_label,pred_field
0,title: privacypreserving retrievalaugmented ge...,5,Cryptography,5,Cryptography
1,title: complementation of emersonlei automata ...,12,Formal Languages and Automata Theory,12,Formal Languages and Automata Theory
2,title: perlin noise improve adversarial robust...,15,General Artificial Intelligence,5,Cryptography
3,title: big data is not the new oil common misc...,7,Database Systems,8,Digital Libraries
4,title: comprehensive movie recommendation syst...,18,General Information Retrieval,18,General Information Retrieval


In [ ]:
comparison = pd.DataFrame([
    {
        "Task": "Discipline",
        "Accuracy": discipline_run["eval_results"].get("eval_accuracy"),
        "Macro Precision": discipline_run["eval_results"].get("eval_macro_precision"),
        "Macro Recall": discipline_run["eval_results"].get("eval_macro_recall"),
        "Macro F1": discipline_run["eval_results"].get("eval_macro_f1"),
        "Weighted F1": discipline_run["eval_results"].get("eval_weighted_f1"),
    },
    {
        "Task": "Field",
        "Accuracy": field_run["eval_results"].get("eval_accuracy"),
        "Macro Precision": field_run["eval_results"].get("eval_macro_precision"),
        "Macro Recall": field_run["eval_results"].get("eval_macro_recall"),
        "Macro F1": field_run["eval_results"].get("eval_macro_f1"),
        "Weighted F1": field_run["eval_results"].get("eval_weighted_f1"),
    }
])

comparison

,Task,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Discipline,0.659471,0.657086,0.658640,0.655756,0.656353
1,Field,0.581907,0.569593,0.582348,0.572160,0.571689


In [ ]:
import os

os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)

comparison.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/scibert_results.csv",
    index=False
)